# ASTR 457: Foundations of Data Science in Astronomy

**Fall 2026, University of Illinois Urbana-Champaign**

Prof. Gautham Narayan | TA: Abha Vishwakarma

Thu Nov 12, 2026 - Day 22: Rare classes, missing data, anomalies, and unsupervised learning

<img src="images/uiuc_logo.png" alt="University of Illinois Urbana-Champaign wordmark" width="220" style="display:block;margin:0 auto;">

In [ ]:
# Presenter setup (a RISE "skip" cell: it runs, but is not a slide). About 40 s.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, IsolationForest
from sklearn.preprocessing import QuantileTransformer
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from ipywidgets import interactive_output, IntSlider, SelectionSlider, HBox, VBox, Layout
from IPython.display import display
plt.rcParams.update({'font.size': 13, 'figure.figsize': (9.4, 3.6)})   # 9.4 in fits 1280 px; 7.4 in beside widget controls

import threading                                    # lock matplotlib's math-text parser (slider redraws run on another thread)
try:
    from matplotlib import _mathtext
    if not getattr(_mathtext.Parser.parse, '_locked', False):
        _mathtext_lock = threading.Lock()
        _unlocked_parse = _mathtext.Parser.parse
        def _locked_parse(self, *args, **kwargs):
            with _mathtext_lock:
                return _unlocked_parse(self, *args, **kwargs)
        _locked_parse._locked = True
        _mathtext.Parser.parse = _locked_parse
except (ImportError, AttributeError):
    print("could not lock matplotlib's math-text parser: do not move a slider while another cell is running")

def controls_right(fn, **widgets):
    for w in widgets.values():
        w.layout = Layout(width='230px')
        w.style = {'description_width': '95px'}
    display(HBox([interactive_output(fn, widgets), VBox(list(widgets.values()), layout=Layout(justify_content='center', width='240px'))]))

# ---- 1. ELAsTiCC (DESC's simulated Rubin alerts). Rare classes: DESC's ELAsTiCC kilonovae among its type Ia and II supernovae, 10 real-time features (data up to 3 d after the first detection, >= 2 detections)
# (most kilonovae get 1-3 detections in all bands together, so Tuesday's 84 features can't be computed for them; all three
#  classes go through the same feature code, built 2026-11-11). A missing value (no observation in that window) is -99.
kn = pd.read_csv('data/day22_elasticc_kn.csv.bz2')
kn_type = kn.pop('type').values
K_IS = kn_type == 'KN'
KX_tr, KX_te, ky_tr, ky_te = train_test_split(kn.fillna(-99).values, K_IS, test_size=0.4, random_state=0, stratify=K_IS)
N_SN = 5000                                         # supernovae in every training set; kilonovae added to make a fraction f
FRACS = [0.3, 0.1, 0.03, 0.01]
def make_rare(f, seed):
    g = np.random.default_rng(seed)
    k = np.r_[g.choice(np.where(ky_tr)[0], int(round(f / (1 - f) * N_SN)), replace=False), g.choice(np.where(~ky_tr)[0], N_SN, replace=False)]
    return KX_tr[k], ky_tr[k]

def rates(p, thresh):                               # completeness on every test kilonova, false-positive rate on the test supernovae
    called = p > thresh
    return called[ky_te].mean(), called[~ky_te].mean()

def purity(tpr, fpr, f):                            # purity on a sky where a fraction f are kilonovae
    return f * tpr / max(f * tpr + (1 - f) * fpr, 1e-12)

METHODS = ['as is', 'reweighted', 'undersampled', 'threshold 0.2']
rare = {}                                           # (method, f) -> (completeness, false-positive rate), mean of 3 draws
for f in FRACS:
    acc = {m: [] for m in METHODS}
    for seed in range(3):
        Xa, ya = make_rare(f, seed)
        p = RandomForestClassifier(150, random_state=seed, n_jobs=-1).fit(Xa, ya).predict_proba(KX_te)[:, 1]
        acc['as is'].append(rates(p, 0.5)); acc['threshold 0.2'].append(rates(p, 0.2))
        p = RandomForestClassifier(150, class_weight='balanced', random_state=seed, n_jobs=-1).fit(Xa, ya).predict_proba(KX_te)[:, 1]
        acc['reweighted'].append(rates(p, 0.5))
        keep = np.r_[np.where(ya)[0], np.random.default_rng(seed).choice(np.where(~ya)[0], ya.sum(), replace=False)]   # as many supernovae as kilonovae
        p = RandomForestClassifier(150, random_state=seed, n_jobs=-1).fit(Xa[keep], ya[keep]).predict_proba(KX_te)[:, 1]
        acc['undersampled'].append(rates(p, 0.5))
    for m in METHODS:
        rare[m, f] = tuple(np.mean(acc[m], axis=0))
N_KN_1PC = int(round(0.01 / 0.99 * N_SN))
kn_ex = pd.read_csv('data/day22_elasticc_kn_examples.csv')   # forced photometry of a type Ia, a kilonova, an SLSN and a PISN
fs_auc = pd.read_csv('data/day22_elasticc_featureset_auc.csv')   # AUC of each feature set for each rare class (instructor build)

# Tuesday's ELAsTiCC supernovae (84 light-curve features, type Ia or II), for the missing-data slides
df = pd.read_csv('data/day21_elasticc_ia_ii.csv.bz2')
y_all = (df.pop('type') == 'Ia').values             # True = type Ia
X_tr, X_te, y_tr, y_te = train_test_split(df, y_all, test_size=0.3, random_state=0)   # Tuesday's split

# ---- missing data: the i-band features (42 of the 84) go missing for 40% of the supernovae
icols = [c for c in df.columns if c.endswith('lssti')]
rcols = [c for c in df.columns if c not in icols]
RMAG = 'mag_weighted_mean_lsstr'                    # mean r-band magnitude: brightness
def missing_mask(X, how, seed):
    if how == 'at random':
        return np.random.default_rng(seed).uniform(size=len(X)) < 0.4
    return (X[RMAG] > X[RMAG].quantile(0.6)).values    # the faintest 40% (in r) lose their i band
STRATS = ['train\nwithout them', 'use the r band\nonly', 'fill with\nthe median', "a booster that\naccepts gaps"]
missing = {}                                        # (how, strategy) -> accuracy on the test supernovae with no i band
for how in ['at random', 'the faintest']:
    mtr, mte = missing_mask(X_tr, how, 1), missing_mask(X_te, how, 2)
    A, B = X_tr.copy(), X_te.copy()
    A.loc[mtr, icols] = np.nan; B.loc[mte, icols] = np.nan
    med = A[icols].median()                         # medians from the training set only
    rf = lambda: RandomForestClassifier(300, random_state=0, n_jobs=-1)
    score = lambda pred: (pred[mte] == y_te[mte]).mean()
    missing[how, STRATS[0]] = score(rf().fit(A[~mtr], y_tr[~mtr]).predict(B.fillna(med)))
    missing[how, STRATS[1]] = score(rf().fit(A[rcols], y_tr).predict(B[rcols]))
    missing[how, STRATS[2]] = score(rf().fit(A.fillna(med), y_tr).predict(B.fillna(med)))
    missing[how, STRATS[3]] = score(HistGradientBoostingClassifier(random_state=0).fit(A, y_tr).predict(B))
    missing[how, 'all'] = score(rf().fit(X_tr, y_tr).predict(X_te))
FAINT_IA = y_te[missing_mask(X_te, 'the faintest', 2)].mean()      # fraction of the faint test supernovae that are Ia

# ---- 2. ZTF DR3, the field around M31: 57,546 r-band light curves, 42 features each (SNAD; Malanchev et al. 2021)
with np.load('data/day22_ztf_m31_features.npz') as a:
    m31_oid, m31_F = a['oid'], a['feature']
m31_Q = QuantileTransformer(output_distribution='uniform', random_state=0).fit_transform(m31_F)   # each feature -> its rank
iso = IsolationForest(n_estimators=1000, max_samples=64, random_state=0).fit(m31_Q)
m31_score = -iso.score_samples(m31_Q)               # 0.5 = typical; near 1 = isolated in very few cuts
lcs = pd.read_csv('data/day22_ztf_m31_lcs.csv')     # ZTF r-band light curves (IRSA, catflags = 0) for 17 of them, by anomaly rank
m31_rank = np.empty(len(m31_score), int); m31_rank[np.argsort(-m31_score)] = np.arange(1, len(m31_score) + 1)
LCS_RANK = {int(m31_rank[np.where(m31_oid == o)[0][0]]): o for o in lcs.oid.unique()}
fmf = pd.read_csv('data/day22_elasticc_fmf.csv')   # light curves: a kilonova found, one missed, a false alarm (undersampled forest at 1%), a faint II
gal = pd.read_csv('data/day22_elasticc_rt_gallery.csv')   # 8 kilonovae, 8 Ia, 8 II: data up to 3 d after the first detection
GAL = list(dict.fromkeys(gal.obj))
# population picture for the methods slide: two of the ten features, 2,000 supernovae and 20 kilonovae (1%), five ways to train
PF = ['peak_mag', 'color_blue_minus_red']
_ok = kn[PF].notna().all(axis=1).values
_g = np.random.default_rng(1)
_sn = _g.choice(np.where(_ok & ~K_IS)[0], 2000, replace=False); _kn = _g.choice(np.where(_ok & K_IS)[0], 20, replace=False)
POP_X = kn[PF].values[np.r_[_sn, _kn]]; POP_Y = np.r_[np.zeros(2000, bool), np.ones(20, bool)]
_keep = np.r_[_g.choice(2000, 20, replace=False), np.arange(2000, 2020)]
_copies = np.r_[np.arange(2000), np.repeat(np.arange(2000, 2020), 100)]
gx, gy = np.meshgrid(np.linspace(19.5, 25.5, 160), np.linspace(-1.5, 1.8, 120))
def _prob(X, y, **kw):
    m = RandomForestClassifier(200, min_samples_leaf=3, random_state=0, n_jobs=-1, **kw).fit(X, y)
    return m.predict_proba(np.c_[gx.ravel(), gy.ravel()])[:, 1].reshape(gx.shape)
POP_P = {'as is': _prob(POP_X, POP_Y), 'reweighted': _prob(POP_X, POP_Y, class_weight='balanced'),
         'undersampled': _prob(POP_X[_keep], POP_Y[_keep]), 'oversampled': _prob(POP_X[_copies], POP_Y[_copies])}
POP_P['threshold 0.2'] = POP_P['as is']
FEAT_END = 58483                                    # 2018-12-31: SNAD's features use DR3 data from 2018 Mar 17 to Dec 31 (Malanchev et al. 2021, sec. 2)

# ---- 3. SDSS: 1,000 stars, 1,000 galaxies and 1,000 quasars with spectra (dereddened ugriz magnitudes)
sdss = pd.read_csv('data/day22_sdss_colors.csv')
ug = (sdss.dered_u - sdss.dered_g).values; gr = (sdss.dered_g - sdss.dered_r).values
ri = (sdss.dered_r - sdss.dered_i).values; iz = (sdss.dered_i - sdss.dered_z).values
ON_AXES = (ug > -0.6) & (ug < 3.6) & (gr > -0.6) & (gr < 2.0) & (ri > -1) & (ri < 2.5) & (iz > -1) & (iz < 2)
sdss, ug, gr, ri, iz = sdss[ON_AXES], ug[ON_AXES], gr[ON_AXES], ri[ON_AXES], iz[ON_AXES]   # drop the 116 with a color off these axes
COL2 = np.c_[ug, gr]                                # the two colors we can draw
COL4 = np.c_[ug, gr, ri, iz]
CLS = sdss['class'].values
KS = list(range(1, 13))
kmeans = {k: KMeans(k, n_init=10, random_state=0).fit(COL2) for k in range(1, 9)}
gmm_bic = [GaussianMixture(k, n_init=3, random_state=0).fit(COL4).bic(COL4) for k in KS]
gmm3 = GaussianMixture(3, n_init=5, random_state=0).fit(COL2)

# ---- 4. SDSS galaxy spectra: 1,000 rest-frame spectra, 1,000 wavelengths each (astroML's spec4000 sample)
with np.load('data/day22_sdss_galaxy_spectra.npz') as a:
    wave, spec, lin_cls = a['wavelength'], a['flux'].astype(float), a['lineindex_cln']
spec = spec / np.median(spec, axis=1, keepdims=True)   # divide out each galaxy's brightness
SPIKY = np.abs(spec).max(axis=1) > 8                # 32 spectra with a line or a glitch above 8x their median: set aside
spec, lin_cls = spec[~SPIKY], lin_cls[~SPIKY]
spec_mean = spec.mean(axis=0)
U, sv, Vt = np.linalg.svd(spec - spec_mean, full_matrices=False)   # rows of Vt = principal components (eigenspectra)
var_frac = sv**2 / np.sum(sv**2)
coef = (spec - spec_mean) @ Vt.T                    # each galaxy's coordinates along the components
EMISSION = lin_cls == 4                             # galaxies with measured emission lines in the star-forming ratio range
I_LINES = int(np.argmax(np.where(EMISSION, spec[:, np.abs(wave - 6563) < 5].max(axis=1), 0)))   # the strongest H-alpha among them
print('setup done')


In [ ]:
# Plotting functions (a RISE "skip" cell). Every function here draws one slide's figure.
C_IA, C_II = 'C0', 'C1'
LINES = {3727: '[OII]', 4861: r'H$\beta$', 5007: '[OIII]', 6563: r'H$\alpha$'}
CLS_COL = {'star': 'C2', 'galaxy': 'C3', 'quasar': 'C4'}
KCOL = ['tab:cyan', 'tab:olive', 'tab:brown', 'tab:pink', 'tab:gray', 'gold', 'navy', 'tab:orange']

def plot_kn_example():
    fig, ax = plt.subplots(1, 2, figsize=(9.4, 2.9), sharey=False)
    for a, (name, g) in zip(ax, [('type Ia', kn_ex[kn_ex.object == 'type Ia']), ('kilonova', kn_ex[kn_ex.object == 'kilonova'])]):
        t0 = g[g.detected == 1].mjd.iloc[np.argmax(g[g.detected == 1].flux.values)]
        w = (g.mjd > t0 - 30) & (g.mjd < t0 + 60)
        for b, c in zip('griz', ['C2', 'C3', 'C1', 'C5']):
            h = g[w & (g.band == b)]
            a.errorbar(h.mjd - t0, h.flux, h.fluxerr, fmt='none', ecolor=c, alpha=0.35)
            a.plot(h.mjd[h.detected == 0] - t0, h.flux[h.detected == 0], 'o', mfc='none', mec=c, ms=5)
            a.plot(h.mjd[h.detected == 1] - t0, h.flux[h.detected == 1], 'o', color=c, ms=7, label=b)
        a.axhline(0, color='0.6', lw=0.8)
        a.set(xlabel='days from the brightest detection', title=f'{name} at redshift {g.z.iloc[0]:.2f}: {int(g[w].detected.sum())} detections' + (', one night' if name == 'kilonova' else ''))
        a.title.set_fontsize(10)
    ax[0].set_ylabel('flux (zero point 27.5 mag)', fontsize=11)
    ax[0].set_ylim(-20, 200); ax[1].set_ylim(-30, 48)
    ax[1].legend(fontsize=9, ncol=4, loc='upper right', title='filter', title_fontsize=9)
    from matplotlib.lines import Line2D
    ax[0].legend([Line2D([], [], ls='', marker='o', color='0.3', ms=7), Line2D([], [], ls='', marker='o', mfc='none', mec='0.3', ms=5)],
                 ['detected (makes an alert)', 'measured, too faint to detect'], fontsize=9, loc='upper right')
    plt.tight_layout(); plt.show()

BAND_COL = {'u': 'C4', 'g': 'C2', 'r': 'C3', 'i': 'C1', 'z': 'C5', 'Y': 'k'}
EX4 = [('type Ia', -30, 60), ('kilonova', -30, 60), ('superluminous SN', -100, 300), ('pair-instability SN', -100, 400)]

def _ex(name):
    g = kn_ex[kn_ex.object == name]
    d = g[g.detected == 1]
    return g, d.mjd.iloc[np.argmax(d.flux.values)], d.mjd.min()

def plot_four_lcs():
    fig, axs = plt.subplots(1, 4, figsize=(9.4, 2.8))
    for a, (name, lo, hi) in zip(axs, EX4):
        g, tp, t1 = _ex(name)
        w = (g.mjd > tp + lo) & (g.mjd < tp + hi)
        for b, c in BAND_COL.items():
            h = g[w & (g.band == b)]
            a.plot(h.mjd[h.detected == 0] - tp, h.flux[h.detected == 0], 'o', mfc='none', mec=c, ms=3)
            a.plot(h.mjd[h.detected == 1] - tp, h.flux[h.detected == 1], 'o', color=c, ms=5)
        a.axhline(0, color='0.6', lw=0.8)
        a.set_title(f'{name}, z = {g.z.iloc[0]:.2f}\n{int(g[w].detected.sum())} detections', fontsize=10)
        a.set_xlabel('days from peak', fontsize=10); a.tick_params(labelsize=9)
    axs[0].set_ylabel('flux', fontsize=10)
    from matplotlib.lines import Line2D
    axs[3].legend([Line2D([], [], ls='', marker='o', color=c, ms=5) for c in BAND_COL.values()], list(BAND_COL), fontsize=7, ncol=2, title='filled: detected', title_fontsize=7, loc='center right')
    plt.tight_layout(); plt.show()

def plot_first3days():
    fig, axs = plt.subplots(1, 4, figsize=(9.4, 2.6), sharey=True)
    for a, (name, lo, hi) in zip(axs, EX4):
        g, tp, t1 = _ex(name)
        w = (g.mjd > t1 - 20) & (g.mjd <= t1 + 3)
        peak = g[w & (g.detected == 1)].flux.max()
        for b, c in BAND_COL.items():
            h = g[w & (g.band == b)]
            a.plot(h.mjd[h.detected == 0] - t1, h.flux[h.detected == 0] / peak, 'o', mfc='none', mec=c, ms=4)
            a.plot(h.mjd[h.detected == 1] - t1, h.flux[h.detected == 1] / peak, 'o', color=c, ms=6)
        a.axvspan(0, 3, color='0.9', zorder=0)
        a.axhline(0, color='0.6', lw=0.8)
        a.set(xlim=(-20, 4), ylim=(-0.6, 1.3))
        a.set_title(f'{name}: {int(g[w].detected.sum())} detections', fontsize=9)
        a.set_xlabel('days from first detection', fontsize=9); a.tick_params(labelsize=9)
    axs[0].set_ylabel('flux / brightest so far', fontsize=9)
    plt.tight_layout(); plt.show()

def plot_rt_features():
    fig, axs = plt.subplots(1, 2, figsize=(9.4, 2.7))
    for a, col, lab, bins in [(axs[0], 'frac_before_3_15', 'flux 3-15 days before the brightest point,\nas a fraction of it', np.linspace(-0.5, 1.5, 41)),
                              (axs[1], 'max_fade_rate', 'fastest fade between two detections (mag / day)', np.linspace(-0.6, 1.2, 37))]:
        v = kn[col].values
        for m, name, c in [(~K_IS, 'supernovae (Ia and II)', '0.5'), (K_IS, 'kilonovae', 'C3')]:
            x = v[m & np.isfinite(v)]
            a.hist(x, bins, density=True, histtype='stepfilled' if c == '0.5' else 'step', color=c, lw=2, alpha=0.6 if c == '0.5' else 1,
                   label=f'{name}: {np.isnan(v[m]).mean():.0%} have no value')
        a.set_xlabel(lab, fontsize=10); a.set_yticks([]); a.legend(fontsize=8.5, loc='upper right')
    plt.tight_layout(); plt.show()

def plot_resample():
    fig, ax = plt.subplots(figsize=(9.4, 2.3))
    n_kn = N_KN_1PC
    groups = [('as is', N_SN, n_kn, ''), ('reweighted', N_SN, n_kn, 'each kilonova\ncounts ~98 times'), ('undersampled', n_kn, n_kn, '')]
    x = np.arange(3)
    ax.bar(x - 0.18, [g[1] for g in groups], 0.36, color='0.6', label='supernovae')
    ax.bar(x + 0.18, [g[2] for g in groups], 0.36, color='C3', label='kilonovae')
    ax.bar(x[1] + 0.18, N_SN, 0.36, fill=False, edgecolor='C3', hatch='//', lw=0)
    for xi, g in zip(x, groups):
        ax.text(xi - 0.18, g[1] * 1.2, f'{g[1]:,}', ha='center', fontsize=10)
        ax.text(xi + 0.18, g[2] * 1.2, f'{g[2]:,}', ha='center', fontsize=10)
    ax.text(x[1] + 0.42, 600, groups[1][3], fontsize=9, color='C3', va='center')
    ax.set_yscale('log'); ax.set_ylim(10, 2e4)
    ax.set_xticks(x, [g[0] for g in groups]); ax.set_ylabel('objects in the\ntraining set (log)', fontsize=10)
    ax.legend(fontsize=9, loc='upper right', ncol=2)
    plt.tight_layout(); plt.show()

def plot_featureset_auc():
    sets = [('84 recomputed, >=5 in r and i', "whole light curve, Tuesday's 84 features", '0.75'),
            ('10 retrospective, >=2 det', 'whole light curve, 10 features', '0.45'),
            ('10 real-time (3 d), >=2 det', 'first 3 days, 10 features', 'C3')]
    rares = [('KN', 'kilonovae'), ('SLSN', 'superluminous SNe'), ('PISN', 'pair-instability SNe')]
    fig, ax = plt.subplots(figsize=(9.4, 2.7))
    x = np.arange(3)
    for j, (key, lab, c) in enumerate(sets):
        v = [fs_auc[(fs_auc.set == key) & (fs_auc.rare == r)].auc for r, _ in rares]
        v = [float(s.iloc[0]) if len(s) else np.nan for s in v]
        ax.bar(x + (j - 1) * 0.27, np.nan_to_num(v, nan=0.5), 0.27, color=c, label=lab, bottom=0)
        for xi, vi in zip(x, v):
            ax.text(xi + (j - 1) * 0.27, (vi if np.isfinite(vi) else 0.5) + 0.01, f'{vi:.2f}' if np.isfinite(vi) else 'none\npass', ha='center', fontsize=9)
    ax.axhline(0.5, color='k', lw=0.8, ls=':')
    ax.set_xticks(x, [r[1] for r in rares]); ax.set(ylim=(0.5, 1.12), ylabel='AUC against type Ia and II\n(0.5 = guessing)')
    ax.legend(fontsize=9, loc='upper right', ncol=3)
    plt.tight_layout(); plt.show()

def plot_snad():
    fig, ax = plt.subplots(figsize=(9.4, 1.6))
    left = 0
    for n, lab, c in [(188, 'bad data: 188', '0.6'), (66, 'known objects: 66', 'C0'), (23, 'new: 23', 'C3')]:
        ax.barh(0, n, left=left, color=c); ax.text(left + n / 2, 0, lab, ha='center', va='center', fontsize=11, color='w' if c != '0.6' else 'k')
        left += n
    ax.set(xlim=(0, 277), yticks=[], xlabel="SNAD's 277 top outliers, checked by eye")
    plt.tight_layout(); plt.show()

POP_M = ['as is', 'reweighted', 'undersampled', 'threshold 0.2']

def plot_pop(method='as is', save=None):
    fig, ax = plt.subplots(figsize=(7.4, 3.0))
    sn, kno = POP_X[~POP_Y], POP_X[POP_Y]
    if method == 'undersampled':
        ax.plot(sn[:, 0], sn[:, 1], '.', color='0.85', ms=2, label='supernovae dropped')
        k = _keep[_keep < 2000]
        ax.plot(POP_X[k, 0], POP_X[k, 1], 'o', color='0.3', ms=4, label='20 supernovae kept')
    else:
        ax.plot(sn[:, 0], sn[:, 1], '.', color='0.55', ms=2, label='2,000 supernovae')
    if method == 'reweighted':
        ax.scatter(kno[:, 0], kno[:, 1], s=160, facecolor='none', edgecolor='C3', lw=1.2, label='20 kilonovae, each weighted 100x (ring)')
    elif method == 'oversampled':
        j = np.random.default_rng(0).normal(0, 0.06, (100 * len(kno), 2))
        ax.plot(np.repeat(kno[:, 0], 100) + j[:, 0], np.repeat(kno[:, 1], 100) + j[:, 1], '^', color='C3', ms=2, alpha=0.25, label='100 copies of each kilonova')
    ax.plot(kno[:, 0], kno[:, 1], '*', color='C3', ms=9, mec='k', mew=0.4, label='20 kilonovae (1%)')
    lev = 0.2 if method == 'threshold 0.2' else 0.5
    if (POP_P[method] > lev).any():
        ax.contour(gx, gy, POP_P[method], levels=[lev], colors='C3', linewidths=2)
    else:
        ax.text(25.3, 1.6, 'no red line: the forest never says kilonova', ha='right', va='top', fontsize=10, color='C3')
    ax.set(xlim=(19.5, 25.5), ylim=(-1.5, 1.8), xlabel='peak magnitude (in the first 3 days)', ylabel='color near the peak',
           title=f'{method}: red line = where the forest says kilonova')
    ax.title.set_fontsize(11)
    ax.legend(fontsize=8, loc='lower left', markerscale=1.2)
    plt.tight_layout()
    if save: plt.savefig(save, dpi=110)
    plt.show()

def pop_controls():
    controls_right(plot_pop, method=SelectionSlider(options=POP_M, value='as is', description='method'))

def plot_gallery(obj=None, save=None):
    obj = obj or GAL[0]
    g = gal[gal.obj == obj]
    fig, axs = plt.subplots(1, 2, figsize=(7.4, 2.9), gridspec_kw={'width_ratios': [1, 1.1]})
    ok = kn[PF].notna().all(axis=1).values
    ax = axs[0]
    ax.plot(kn.peak_mag.values[ok & ~K_IS][::4], kn.color_blue_minus_red.values[ok & ~K_IS][::4], '.', color='0.7', ms=2, label='supernovae')
    ax.plot(kn.peak_mag.values[ok & K_IS][::2], kn.color_blue_minus_red.values[ok & K_IS][::2], '.', color='C3', ms=2, alpha=0.5, label='kilonovae')
    ax.plot(g.peak_mag.iloc[0], g.color.iloc[0], 'o', mfc='none', mec='k', ms=14, mew=2)
    ax.set(xlim=(19.5, 25.5), ylim=(-1.5, 1.8), xlabel='peak magnitude', ylabel='color near the peak\n(bluer - redder, mag)')
    ax.legend(fontsize=8, loc='lower left', markerscale=4)
    ax = axs[1]
    peak = g[g.detected == 1].flux.max()
    for bnd, c in BAND_COL.items():
        h = g[g.band == bnd]
        ax.plot(h.t[h.detected == 0], h.flux[h.detected == 0] / peak, 'o', mfc='none', mec=c, ms=4)
        ax.plot(h.t[h.detected == 1], h.flux[h.detected == 1] / peak, 'o', color=c, ms=6)
    ax.axvspan(0, 3, color='0.9', zorder=0); ax.axhline(0, color='0.6', lw=0.8)
    name = {'KN': 'kilonova', 'Ia': 'type Ia', 'II': 'type II'}[g.type.iloc[0]]
    ax.set(xlim=(-20, 4), ylim=(-0.8, 1.3), xlabel='days from first detection', ylabel='flux / brightest', title=f'the circled point: a {name}')
    ax.title.set_fontsize(11)
    plt.tight_layout()
    if save: plt.savefig(save, dpi=110)
    plt.show()

def gallery_controls():
    controls_right(plot_gallery, obj=SelectionSlider(options=[(f'{i + 1}', o) for i, o in enumerate(GAL)], value=GAL[0], description='object'))

def plot_m31_pick(rank=1, save=None):
    o = LCS_RANK[rank]; s = m31_score[np.where(m31_oid == o)[0][0]]
    fig, axs = plt.subplots(1, 2, figsize=(7.4, 2.9), gridspec_kw={'width_ratios': [1, 1.3]})
    ax = axs[0]
    ax.hist(m31_score, 80, color='0.6'); ax.set_yscale('log')
    ax.axvline(s, color='C3', lw=2)
    ax.set(xlabel='anomaly score', ylabel='light curves (log)', title=f'rank {rank:,} of 57,546: score {s:.2f}')
    ax.title.set_fontsize(10)
    ax = axs[1]
    g = lcs[lcs.oid == o]
    ax.errorbar(g.mjd, g.mag, g.magerr, fmt='.', ms=2, lw=0.4, color='k', alpha=0.6)
    ax.axvspan(g.mjd.min() - 20, FEAT_END, color='C0', alpha=0.12)
    ax.invert_yaxis(); ax.set(xlabel='MJD', ylabel='r mag')
    plt.tight_layout()
    if save: plt.savefig(save, dpi=110)
    plt.show()

def m31_controls():
    controls_right(plot_m31_pick, rank=SelectionSlider(options=[(f'{r:,}', r) for r in sorted(LCS_RANK)], value=1, description='rank'))

def plot_ladder():
    fig, ax = plt.subplots(figsize=(9.4, 2.4))
    Y, V = '#d9a400', '#7a4fbf'                     # yellow: what we choose; violet: what the method learns
    rows = [("Tuesday's forest", '84 features, chosen by hand', 'the mapping from features to type'),
            ("today's forest", '10 features, chosen by hand', 'the mapping from features to type'),
            ('next Tuesday: neural networks', 'the network and the labels', 'the features themselves, from the light curve')]
    for i, (name, spec, learn) in enumerate(rows):
        y = 2 - i
        ax.text(0.0, y, name, ha='left', va='center', fontsize=11)
        ax.add_patch(plt.Rectangle((0.30, y - 0.35), 0.30, 0.7, color=Y, alpha=0.85))
        ax.text(0.45, y, spec, ha='center', va='center', fontsize=10)
        ax.add_patch(plt.Rectangle((0.62, y - 0.35), 0.38, 0.7, color=V, alpha=0.85))
        ax.text(0.81, y, learn, ha='center', va='center', fontsize=10, color='w')
    ax.text(0.45, 2.75, 'we choose', ha='center', fontsize=11, color=Y, weight='bold')
    ax.text(0.81, 2.75, 'the method learns', ha='center', fontsize=11, color=V, weight='bold')
    ax.set(xlim=(0, 1), ylim=(-0.5, 3.1)); ax.axis('off')
    plt.tight_layout(); plt.show()

def _lc(ax, g, norm=True, tmax=40):
    peak = g[g.detected == 1].flux.max() if norm else 1.0
    for bnd, c in BAND_COL.items():
        h = g[(g.band == bnd) & (g.t <= tmax)]
        ax.plot(h.t[h.detected == 0], h.flux[h.detected == 0] / peak, 'o', mfc='none', mec=c, ms=4)
        ax.plot(h.t[h.detected == 1], h.flux[h.detected == 1] / peak, 'o', color=c, ms=6)
    ax.axhline(0, color='0.6', lw=0.8)

def plot_fmf():
    fig, axs = plt.subplots(1, 3, figsize=(9.4, 2.7), sharey=True)
    names = {'KN': 'kilonova', 'Ia': 'type Ia', 'II': 'type II'}
    for a, panel in zip(axs, ['found', 'missed', 'false alarm']):
        g = fmf[fmf.panel == panel]
        _lc(a, g)
        a.axvspan(0, 3, color='0.9', zorder=0)
        a.set(xlim=(-20, 40), ylim=(-0.8, 1.4), xlabel='days from first detection')
        a.set_title(f'{panel}: a {names[g.type.iloc[0]]}, forest says {g.p_kn.iloc[0]:.2f}', fontsize=10)
    axs[0].set_ylabel('flux / brightest')
    axs[0].text(4, -0.7, 'shaded: what the forest saw', fontsize=8, ha='left', va='bottom')
    plt.tight_layout(); plt.show()

def plot_faint_ii():
    fig, ax = plt.subplots(figsize=(9.4, 2.4))
    g = fmf[fmf.panel == 'faint II']
    for bnd in ['r', 'i']:
        h = g[g.band == bnd]; c = BAND_COL[bnd]
        ax.errorbar(h.t, h.flux, h.fluxerr, fmt='none', ecolor=c, alpha=0.4)
        ax.plot(h.t[h.detected == 0], h.flux[h.detected == 0], 'o', mfc='none', mec=c, ms=5, label=f'{bnd}: below the detection limit')
        ax.plot(h.t[h.detected == 1], h.flux[h.detected == 1], 'o', color=c, ms=7, label=f'{bnd}: detected')
    ax.axhline(0, color='0.6', lw=0.8)
    ax.set(xlabel='days from first detection', ylabel='flux')
    ax.legend(fontsize=9, ncol=2, loc='upper right')
    plt.tight_layout(); plt.show()

def plot_m31_typical():
    fig, axs = plt.subplots(1, 4, figsize=(9.4, 2.2), sharex=True)
    for a, r in zip(axs, [1000, 10000, 28773, 57000]):
        g = lcs[lcs.oid == LCS_RANK[r]]
        a.plot(g.mjd, g.mag, '.', ms=1.5, color='k', alpha=0.5)
        a.invert_yaxis(); a.set_title(f'rank {r:,}', fontsize=10); a.tick_params(labelsize=8)
        a.set_xlabel('MJD', fontsize=9)
    axs[0].set_ylabel('r mag', fontsize=9)
    plt.tight_layout(); plt.show()

def plot_pca_2d():
    s = CLS == 'star'
    X = np.c_[ug[s], gr[s]]; m = X.mean(0)
    w, v = np.linalg.eigh(np.cov((X - m).T)); v = v[:, ::-1]; w = w[::-1]
    fig, axs = plt.subplots(1, 2, figsize=(9.4, 2.8))
    axs[0].plot(X[:, 0], X[:, 1], '.', color='C2', ms=2, alpha=0.5)
    for k, c in enumerate(['C3', 'C0']):
        d = v[:, k] * 2.5 * np.sqrt(w[k])
        axs[0].plot([m[0] - d[0], m[0] + d[0]], [m[1] - d[1], m[1] + d[1]], color=c, lw=2.5)
    axs[0].text(0.03, 0.95, 'component 1', color='C3', fontsize=10, transform=axs[0].transAxes, va='top')
    axs[0].text(0.03, 0.84, 'component 2', color='C0', fontsize=10, transform=axs[0].transAxes, va='top')
    axs[0].set(xlabel='u - g', ylabel='g - r', title='the stars in two colors', aspect='equal')
    Y = (X - m) @ v
    axs[1].plot(Y[:, 0], Y[:, 1], '.', color='C2', ms=2, alpha=0.5)
    axs[1].set(xlabel='component 1', ylabel='component 2', title='the same stars, axes rotated', aspect='equal')
    for a in axs: a.title.set_fontsize(11)
    plt.tight_layout(); plt.show()

def plot_pc1_ends():
    o = np.argsort(coef[:, 0])
    fig, axs = plt.subplots(1, 2, figsize=(9.4, 2.7), sharey=True)
    for a, idx, lab in [(axs[0], o[:3], 'most negative component 1'), (axs[1], o[-3:], 'most positive component 1')]:
        for j, i in enumerate(idx):
            a.plot(wave, spec[i] + 1.5 * j, lw=0.7)
        a.axvline(6563, color='0.7', ls=':', lw=0.8); a.text(6600, 4.6, r'H$\alpha$', fontsize=9)
        a.set(xlabel='rest wavelength (Angstrom)', title=lab); a.title.set_fontsize(11)
    axs[0].set_ylabel('flux / median + offset')
    plt.tight_layout(); plt.show()

def plot_rare_asis():
    fig, ax = plt.subplots(figsize=(9.4, 2.9))
    x = np.arange(len(FRACS))
    comp = [rare['as is', f][0] for f in FRACS]
    pur = [purity(*rare['as is', f], f) for f in FRACS]
    accu = [f * rare['as is', f][0] + (1 - f) * (1 - rare['as is', f][1]) for f in FRACS]
    ax.plot(x, accu, 'k-o', label='accuracy')
    ax.plot(x, pur, 'C3-o', label='purity: fraction of its "kilonovae" that are kilonovae')
    ax.plot(x, comp, 'C0-o', lw=2.5, label='completeness: fraction of the kilonovae it finds')
    ax.set_xticks(x, [f'{f:.0%}' for f in FRACS])
    ax.set(xlabel='kilonovae as a fraction of the training set (and of the sky)', ylabel='fraction', ylim=(0, 1.04))
    ax.legend(fontsize=10, loc='lower left')
    plt.tight_layout(); plt.show()

def plot_rare(frac=0.01, save=None):
    fig, ax = plt.subplots(figsize=(7.4, 3.0))
    x = np.arange(len(METHODS))
    comp = [rare[m, frac][0] for m in METHODS]
    pur = [purity(*rare[m, frac], frac) for m in METHODS]
    ax.bar(x - 0.2, comp, 0.4, color='C0', label='completeness')
    ax.bar(x + 0.2, pur, 0.4, color='C3', label='purity')
    for i in range(len(METHODS)):
        ax.text(x[i] - 0.2, comp[i] + 0.02, f'{comp[i]:.2f}', ha='center', fontsize=10)
        ax.text(x[i] + 0.2, pur[i] + 0.02, f'{pur[i]:.2f}', ha='center', fontsize=10)
    ax.set_xticks(x, METHODS)
    ax.set(ylim=(0, 1.32), ylabel='fraction', title=f'kilonovae = {frac:.0%} of the training set and the sky')
    ax.legend(fontsize=10, loc='upper center', ncol=2)
    plt.tight_layout()
    if save: plt.savefig(save, dpi=110)
    plt.show()

def rare_controls():
    controls_right(plot_rare, frac=SelectionSlider(options=[(f'{f:.0%}', f) for f in FRACS], value=0.01, description='kilonovae'))

def plot_missing_who():
    fig, ax = plt.subplots(figsize=(9.4, 2.7))
    m = missing_mask(X_te, 'the faintest', 2)
    bins = np.linspace(19, 24.5, 45)
    ax.hist(X_te[RMAG][~m & y_te], bins, histtype='step', color=C_IA, lw=2, label='Ia, has i band')
    ax.hist(X_te[RMAG][~m & ~y_te], bins, histtype='step', color=C_II, lw=2, label='II, has i band')
    ax.hist([X_te[RMAG][m & ~y_te], X_te[RMAG][m & y_te]], bins, stacked=True, color=['0.55', '0.8'], label=[f'II, no i band ({1 - FAINT_IA:.0%})', f'Ia, no i band ({FAINT_IA:.0%})'])
    ax.set(xlabel='mean r-band magnitude', ylabel='test supernovae')
    ax.legend(fontsize=10, loc='upper left')
    plt.tight_layout(); plt.show()

def plot_missing():
    fig, ax = plt.subplots(figsize=(9.4, 2.9))
    x = np.arange(len(STRATS))
    for j, (how, col) in enumerate([('at random', '0.6'), ('the faintest', 'C3')]):
        v = [missing[how, s] for s in STRATS]
        ax.bar(x + (j - 0.5) * 0.38, v, 0.38, color=col, label=f'i band missing for 40%, {how}')
        for i in range(len(x)):
            ax.text(x[i] + (j - 0.5) * 0.38, v[i] + 0.01, f'{v[i]:.2f}', ha='center', fontsize=10)
    ax.axhline(1 - FAINT_IA, color='C3', ls='--', lw=1)
    ax.text(0.5, 1 - FAINT_IA - 0.012, f'call every faint one type II: {1 - FAINT_IA:.2f}', ha='center', va='top', fontsize=10, color='C3', bbox=dict(facecolor='white', edgecolor='none', pad=1))
    ax.set_xticks(x, STRATS, fontsize=11)
    ax.set(ylim=(0.5, 1.16), ylabel='accuracy on the supernovae\nmissing their i band (axis from 0.5)')
    ax.legend(fontsize=10, loc='upper center', ncol=2)
    plt.tight_layout(); plt.show()

# ---- isolation: random cuts in 2-D, following one typical point and one outlier
_rng = np.random.default_rng(3)
TOY = np.r_[_rng.normal(0, 1, (250, 2)), [[3.6, -3.2]]]
WHO = {'typical point': 0, 'outlier': len(TOY) - 1}
TOY[0] = [0.15, 0.1]

def isolation_path(i, seed):
    rng = np.random.default_rng(seed)
    box = np.array([TOY.min(0) - 0.3, TOY.max(0) + 0.3])  # [[xmin, ymin], [xmax, ymax]]
    inside = np.ones(len(TOY), bool); cuts = []
    while inside.sum() > 1 and len(cuts) < 40:
        j = rng.integers(2)
        lo, hi = TOY[inside, j].min(), TOY[inside, j].max()
        c = rng.uniform(lo, hi)
        cuts.append((j, c, box.copy()))
        if TOY[i, j] < c:
            box[1, j] = c; inside &= TOY[:, j] < c
        else:
            box[0, j] = c; inside &= TOY[:, j] >= c
    return cuts, box

PATHS = {k: isolation_path(i, 11) for k, i in WHO.items()}
DEPTHS = {k: [len(isolation_path(i, s)[0]) for s in range(200)] for k, i in WHO.items()}

def plot_isolation(n_cuts=3, save=None):
    fig, axs = plt.subplots(1, 2, figsize=(7.4, 3.0), sharex=True, sharey=True)
    for ax, (k, i) in zip(axs, WHO.items()):
        cuts, final = PATHS[k]
        ax.plot(*TOY.T, '.', color='0.6', ms=3)
        for j, c, b in cuts[:n_cuts]:
            if j == 0: ax.plot([c, c], [b[0, 1], b[1, 1]], 'k-', lw=0.8)
            else: ax.plot([b[0, 0], b[1, 0]], [c, c], 'k-', lw=0.8)
        ax.plot(*TOY[i], 'r*', ms=14)
        done = n_cuts >= len(cuts)
        ax.set_title(f'{k}: ' + (f'alone after {len(cuts)} cuts' if done else f'{n_cuts} of {len(cuts)} cuts'), fontsize=11)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set(xlim=(TOY[:, 0].min() - 0.5, TOY[:, 0].max() + 0.5), ylim=(TOY[:, 1].min() - 0.5, TOY[:, 1].max() + 0.5))
    plt.tight_layout()
    if save: plt.savefig(save, dpi=110)
    plt.show()

def isolation_controls():
    controls_right(plot_isolation, n_cuts=IntSlider(value=3, min=0, max=max(len(p[0]) for p in PATHS.values()), description='random cuts'))

def plot_depths():
    fig, ax = plt.subplots(figsize=(9.4, 2.9))
    bins = np.arange(0, 26) - 0.5
    for k, col in [('typical point', 'C0'), ('outlier', 'C3')]:
        ax.hist(DEPTHS[k], bins, color=col, alpha=0.6, label=f'{k}: mean {np.mean(DEPTHS[k]):.1f} cuts')
    ax.set(xlabel='random cuts needed to isolate the point (200 random trees)', ylabel='trees')
    ax.legend(fontsize=10)
    plt.tight_layout(); plt.show()

def plot_scores():
    fig, ax = plt.subplots(figsize=(9.4, 2.7))
    ax.hist(m31_score, 120, color='0.5')
    ax.set_yscale('log')
    top = np.sort(m31_score)[-100]
    ax.axvline(top, color='C3', ls='--')
    ax.text(top + 0.004, 3000, 'top 100', color='C3', fontsize=11)
    ax.set(xlabel='anomaly score (0.5 = typical, higher = isolated in fewer cuts)', ylabel='light curves (log scale)')
    plt.tight_layout(); plt.show()

def plot_m31_lcs():
    rank = {LCS_RANK[r]: r for r in (1, 2, 6, 28773)}
    order = [LCS_RANK[r] for r in (1, 2, 6, 28773)]
    fig, axs = plt.subplots(2, 2, figsize=(9.4, 2.9))
    for ax, o in zip(axs.flat, order):
        g = lcs[lcs.oid == o]
        ax.errorbar(g.mjd, g.mag, g.magerr, fmt='.', ms=2, lw=0.4, color='k', alpha=0.6)
        ax.axvspan(g.mjd.min() - 20, FEAT_END, color='C0', alpha=0.12)
        ax.invert_yaxis()
        s = m31_score[np.where(m31_oid == o)[0][0]]
        ax.set_title(f'rank {rank[o]:,} of 57,546 (score {s:.2f})', fontsize=11)
        ax.tick_params(labelsize=9)
    for ax in axs[1]: ax.set_xlabel('MJD', fontsize=11)
    for ax in axs[:, 0]: ax.set_ylabel('r mag', fontsize=11)
    plt.tight_layout(); plt.show()

def plot_sdss_gray():
    fig, ax = plt.subplots(figsize=(9.4, 2.9))
    ax.plot(ug, gr, '.', color='0.3', ms=2, alpha=0.5)
    ax.set(xlabel='u - g', ylabel='g - r', xlim=(-0.6, 3.6), ylim=(-0.6, 2.0))
    plt.tight_layout(); plt.show()

def plot_kmeans(k=3, save=None):
    fig, axs = plt.subplots(1, 2, figsize=(7.4, 3.0), sharex=True, sharey=True)
    km = kmeans[k]
    rank = np.argsort(np.argsort(km.cluster_centers_[:, 0]))      # number clusters from blue to red in u - g
    axs[0].scatter(ug, gr, c=[KCOL[rank[l]] for l in km.labels_], s=2, alpha=0.5)
    axs[0].plot(*km.cluster_centers_.T, 'kX', ms=10)
    axs[0].set_title(f'k-means, k = {k}', fontsize=11)
    for c in ['galaxy', 'quasar', 'star']:                          # stars drawn last, so their band shows
        col, m = CLS_COL[c], CLS == c
        axs[1].plot(ug[m], gr[m], '.', color=col, ms=2, alpha=0.5, label=c)
    axs[1].set_title('the classes from their spectra', fontsize=11)
    axs[1].legend(fontsize=10, markerscale=5, loc='upper left')
    for ax in axs: ax.set(xlabel='u - g', xlim=(-0.6, 3.6), ylim=(-0.6, 2.0))
    axs[0].set_ylabel('g - r')
    plt.tight_layout()
    if save: plt.savefig(save, dpi=110)
    plt.show()

def kmeans_controls():
    controls_right(plot_kmeans, k=IntSlider(value=3, min=1, max=8, description='clusters k'))

def plot_kmeans_table():
    km = kmeans[3]
    rank = np.argsort(np.argsort(km.cluster_centers_[:, 0]))
    tab = pd.crosstab(rank[km.labels_], CLS)[['star', 'galaxy', 'quasar']]
    fig, ax = plt.subplots(figsize=(9.4, 2.7))
    ax.imshow(tab.values, cmap='Greys', vmax=1400, aspect=0.45)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, tab.values[i, j], ha='center', va='center', fontsize=16, color='w' if tab.values[i, j] > 700 else 'k')
    ax.set_xticks(range(3), ['stars', 'galaxies', 'quasars'])
    ax.set_yticks(range(3), [f'{n} cluster' for n in ['cyan (bluest)', 'olive', 'brown (reddest)']])
    ax.tick_params(labelsize=13)
    ax.set_title('number of objects in each k-means cluster, by class from the spectrum', fontsize=12)
    plt.tight_layout(); plt.show()

def plot_gmm():
    from matplotlib.patches import Ellipse
    fig, axs = plt.subplots(1, 2, figsize=(9.4, 2.9), gridspec_kw={'width_ratios': [1.3, 1]})
    ax = axs[0]
    ax.plot(ug, gr, '.', color='0.4', ms=2, alpha=0.4)
    for mu_k, cov in zip(gmm3.means_, gmm3.covariances_):
        vals, vecs = np.linalg.eigh(cov)
        ang = np.degrees(np.arctan2(vecs[1, 1], vecs[0, 1]))
        for n in (1, 2):
            ax.add_patch(Ellipse(mu_k, 2 * n * np.sqrt(vals[1]), 2 * n * np.sqrt(vals[0]), angle=ang, fill=False, color='C3', lw=1.5))
    ax.set(xlabel='u - g', ylabel='g - r', xlim=(-0.6, 3.6), ylim=(-0.6, 2.0), title='3 Gaussians (1 and 2 sigma)')
    axs[1].plot(KS, gmm_bic, 'k-o')
    axs[1].set(xlabel='number of Gaussians', ylabel='BIC (lower is better)', title='all four colors', xticks=range(2, 13, 2))
    plt.tight_layout(); plt.show()

def plot_spectra():
    fig, ax = plt.subplots(figsize=(9.4, 2.7))
    for i, off in zip([0, 3, 7, 12], [0, 1.2, 2.4, 3.6]):
        ax.plot(wave, spec[i] + off, lw=0.8)
    for lam, name in LINES.items():
        ax.axvline(lam, color='0.7', ls=':', lw=0.8, zorder=0)
        ax.text(lam + (15 if lam != 4861 else -15), ax.get_ylim()[1] * 0.97, name, fontsize=10, va='top', ha='left' if lam != 4861 else 'right')
    ax.set(xlabel='rest wavelength (Angstrom)', ylabel='flux / median + offset')
    plt.tight_layout(); plt.show()

def plot_eigenspectra():
    fig, axs = plt.subplots(1, 2, figsize=(9.4, 2.7), gridspec_kw={'width_ratios': [1.5, 1]})
    axs[0].plot(wave, spec_mean / spec_mean.max(), 'k', lw=1)
    axs[0].text(7950, spec_mean[-1] / spec_mean.max(), 'mean', fontsize=10, va='center')
    for j in range(3):
        v = Vt[j] / np.abs(Vt[j]).max()
        axs[0].plot(wave, v * 0.5 - 1.1 * (j + 1), lw=0.8, color=f'C{j}')
        axs[0].text(7950, -1.1 * (j + 1), f'{j + 1}', fontsize=11, color=f'C{j}', va='center')
    axs[0].axvline(6563, color='0.7', ls=':', lw=0.8)
    axs[0].text(6600, 1.05, r'H$\alpha$', fontsize=10)
    axs[0].set(xlabel='rest wavelength (Angstrom)', yticks=[], xlim=(3100, 8500), title='the mean and components 1-3 (each rescaled)')
    axs[1].plot(np.arange(1, 31), np.cumsum(var_frac)[:30], 'k-o', ms=3)
    axs[1].set(xlabel='number of components', ylabel='fraction of variance\n(cumulative)', ylim=(0, 1.02))
    plt.tight_layout(); plt.show()

def plot_reconstruct(n=2, save=None):
    fig, ax = plt.subplots(figsize=(7.4, 3.0))
    i = I_LINES
    rec = spec_mean + coef[i, :n] @ Vt[:n]
    ax.plot(wave, spec[i], color='0.6', lw=1.2, label='a galaxy (1,000 numbers)')
    ax.plot(wave, rec, 'C3', lw=1, label=f'mean + {n} components ({n} numbers)')
    ax.set(xlabel='rest wavelength (Angstrom)', ylabel='flux / median', ylim=(0, spec[i].max() * 1.15))
    ax.legend(fontsize=10, loc='upper left')
    plt.tight_layout()
    if save: plt.savefig(save, dpi=110)
    plt.show()

def reconstruct_controls():
    controls_right(plot_reconstruct, n=SelectionSlider(options=[0, 1, 2, 3, 5, 10, 30], value=2, description='components'))

def plot_pc12():
    fig, ax = plt.subplots(figsize=(9.4, 2.9))
    ax.plot(coef[~EMISSION, 0], coef[~EMISSION, 1], '.', color='C3', ms=3, alpha=0.6, label='other galaxies')
    ax.plot(coef[EMISSION, 0], coef[EMISSION, 1], '.', color='C0', ms=3, alpha=0.6, label='star-forming emission lines')
    ax.set(xlabel='coefficient of component 1', ylabel='coefficient of component 2', xlim=(-16, 10))
    ax.legend(fontsize=10, markerscale=4)
    plt.tight_layout(); plt.show()


## Quiz 5 is the first 15 minutes

* closed everything: pens only, no notes, no phone, no laptop
* 15 minutes, 10 points, 3 questions: ABC (Nov 3) and Tuesday's machine learning
* write on the sheet, put your name on it, and hand it in when I call time

**Time starts when everyone has a sheet.**

## Recap - supervised learning needs labeled data

<img src="images/day21_static_knn_widget.png" alt="Tuesday's nearest-neighbor widget: type Ia (blue) and type II (orange) supernovae in two features, with the 15-neighbor decision regions; with the features on their raw scales the regions are stripes, after rescaling they follow the data" style="display:block;margin:0 auto;max-height:250px;max-width:100%">

* Tuesday: features on very different scales broke nearest neighbors - you said take logs, or standardize
* with labeled data, forests and boosting got 99% of new supernovae right

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**What do we do when the labels are rare, incomplete or absent?**

</div>

## Today - rare classes, missing data, and no labels at all

* rare classes: how do you pick out a kilonova among 100 supernovae, by day 3?
* missing data: what if some supernovae have no i band?
* no labels: which of 57,546 light curves is the odd one out, and can a method find groups by itself?
* last 10 minutes: **Celeste Emery** and **Ben Hall-Skank** defend Lab 08

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**LOOK AT YOUR DATA before you believe what any method finds, with or without labels!**

</div>

## What if the class you want is 1% of the data?

* two neutron stars merge, and the radioactive heavy elements they throw out glow for days - a **kilonova**
* a **detection** is a measurement bright enough to make an alert, and ELAsTiCC (Tuesday's simulated Rubin alerts) detects only 8,122 of its 416,696 kilonovae
* that's about 1.5 kilonovae per 100 detected supernovae, so we'll call them 1% of the data

<img src="images/day20_eso1733_kilonova_pair.jpg" alt="Left: an artist's impression of two merging neutron stars. Right: spectra of AT2017gfo, the kilonova after GW170817, one per night from 0.5 to 11.5 days, fading and reddening" style="display:block;margin:0 auto;max-height:180px;max-width:100%">

<div style="font-size:0.8em; text-align:center; color:#666;">Left: merging neutron stars (ESO/L. Calçada/M. Kornmesser). Right: spectra of AT2017gfo, the kilonova after GW170817, one per night (Tanvir et al., ESO). eso1733, CC BY 4.0.</div>

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Most kilonovae fade before Rubin detects them.**

</div>

## A kilonova fades much faster than a type Ia

In [ ]:
# a typical ELAsTiCC type Ia and kilonova: flux in each filter against time
plot_kn_example()

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**This kilonova was detected on a single night.**

</div>

## Superluminous and pair-instability supernovae last for months

* a **superluminous supernova** (SLSN) is 10 to 100 times brighter than an ordinary one (this one is farther away, at redshift 0.54)
* a **pair-instability supernova** (PISN) is the predicted explosion of a star born with more than about 140 solar masses (none confirmed yet)

In [ ]:
# one ELAsTiCC example of each (filled: detected, open: below the detection limit)
plot_four_lcs()

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**These rare classes last anywhere from one night to several months.**

</div>

## Real-time classification uses only what Rubin has seen so far

* a kilonova needs a spectrum within days, so we must decide by day 3 after its first detection (shaded)
* Tuesday's 84 features need 5 detections in both r and i - no kilonova has that, so we use only what Rubin has measured by day 3 (open points: measured, not detected)

In [ ]:
# the same four objects, as Rubin has them 3 days after the first detection
plot_first3days()

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**We can only use what Rubin has measured by the time we must decide.**

</div>

## A point in feature space is one light curve

* slide through 24 objects: the circle marks where each light curve lands in two of the ten features
* the left panel shows every object, not the sky's 1%, and the two classes overlap in these two features

In [ ]:
# left: two features for every object; right: the circled object's light curve, as Rubin has it on day 3
gallery_controls()

<img src="images/day22_static_gallery_widget.png" alt="The gallery widget: left, peak magnitude against color near the peak for supernovae (gray) and kilonovae (red), with one kilonova circled; right, that kilonova's light curve up to 3 days after its first detection" style="max-height:330px; display:block; margin:0 auto;">

(static copy of the widget, for readers on GitHub or without running the notebook)

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Look at the light curves behind the points before you trust a feature.**

</div>

## We compute the same 10 features for every class

* 7 from the detections: count, filters, days spanned, peak, signal-to-noise, color, fastest fade
* 3 from all measurements: the flux before the peak, the latest flux after it, and how many days later that was
* a feature has no value when there's no measurement in its window - and computing them differently for one class would let the forest learn our code instead of the physics

In [ ]:
# two of the ten, for 3,624 kilonovae and 10,275 type Ia and II with at least 2 detections by day 3
plot_rt_features()

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**A kilonova fades faster than a supernova and shows nothing before its peak.**

</div>

## A forest trained on 1% kilonovae finds 1 in 20

* each training set has 5,000 supernovae plus kilonovae at 30%, 10%, 3% or 1% (51 at 1%), and purity is for a sky with that same fraction
* at 1% the forest calls almost nothing a kilonova, so the purity rests on very few objects

In [ ]:
# a 150-tree forest trained on 5,000 supernovae plus kilonovae at 30%, 10%, 3% and 1% of the training set
plot_rare_asis()

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**A 99% accurate forest still misses 95% of the kilonovae!**

</div>

## You can reweight, resample, or move the threshold

* **reweight**: in training, a missed kilonova counts about 100 times as much as a missed supernova
* **resample**: drop supernovae (**undersampling**) or copy kilonovae (**oversampling**)
* **move the threshold**: call it a kilonova when 20% of the trees vote for one (the default is 50%)

In [ ]:
# slide the method: 2,000 supernovae, 20 kilonovae, and where a forest on two features says kilonova
pop_controls()

<img src="images/day22_static_pop_widget.png" alt="The population widget, reweighted: 2,000 supernovae as gray dots and 20 kilonovae as red stars drawn larger for their weight, with the red line where a forest on peak magnitude and color says kilonova" style="max-height:330px; display:block; margin:0 auto;">

(static copy of the widget, for readers on GitHub or without running the notebook)

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Each method moves the red line, but none adds information about kilonovae.**

</div>

## No method raises completeness without losing purity

In [ ]:
# slide the kilonova fraction: completeness and purity for the four choices (mean of 3 training sets)
rare_controls()

<img src="images/day22_static_rare_widget.png" alt="The rare-class widget at 1% kilonovae: completeness and purity bars for the forest as is (0.05 and 0.86), reweighted (0.06 and 0.36), undersampled (0.72 and 0.03), and with the threshold at 0.2 (0.12 and 0.25)" style="max-height:330px; display:block; margin:0 auto;">

(static copy of the widget, for readers on GitHub or without running the notebook)

* at 1%, reweighting barely helps (5% to 6% found), and undersampling finds 72% but only 3% of what it calls a kilonova is one
* at 30%, the best completeness is 0.86, at purity 0.53

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**You need to decide whether a missed kilonova or a false alarm is worse for your science!**

</div>

## The undersampled forest's mistakes are light curves too

* one kilonova it found, one it missed, and a type Ia it called a kilonova (shaded: the first 3 days, all the forest saw)

In [ ]:
# three test objects and the undersampled forest's kilonova score (trained at 1%)
plot_fmf()

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Look at the light curves of the mistakes before you trust a purity or a completeness.**

</div>

## What we can measure decides which rare class we recognize

* about half the SLSNe and PISNe have 5 detections in both r and i, so their light-gray bars use only those
* each forest is trained with 10% rare and scored by AUC, which covers every threshold at once (Tuesday's ROC)
* in its first 3 days a slowly rising SLSN looks like any supernova

In [ ]:
# AUC for each rare class against type Ia and II (forests trained with 10% rare)
plot_featureset_auc()

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**With only 3 days of data, superluminous supernovae barely stand out.**

</div>

## What if some supernovae are missing a band?

* many of our 10 kilonova features had no value - Tuesday's supernovae let us test what to do about gaps
* below, a faint type II that Rubin detects in r but never in i

In [ ]:
# one ELAsTiCC type II: r detected, i measured but never detected
plot_faint_ii()

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**A band can be missing because the object is too faint in it.**

</div>

## Missing because faint is not missing at random

* remove the i-band features for 40% of Tuesday's supernovae, either at random (**missing completely at random**) or for the faintest 40% (**missing because faint**)

In [ ]:
# r-band brightness of the test supernovae, with an i band and without (the faintest 40%)
plot_missing_who()

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Note how 86% of the faint supernovae with no i band are type II (type II are fainter than Ia, as we saw Tuesday)!**

</div>

## Dropping the faint supernovae ruins the classifier

In [ ]:
# accuracy on the supernovae with no i band, four ways (the booster: HistGradientBoostingClassifier)
plot_missing()

* a missing i band is **censoring** (the value is missing) - dropping the supernova makes it **truncation** (the object is missing, Day 19)
* a forest trained without them gets 60% of them right (calling them all II gets 86%), while median filling (**imputation**) gets 97%

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Before you drop an object with a gap, ask why the gap is there!**

</div>

## What if we don't know what we're looking for?

* ZTF (the Zwicky Transient Facility) has imaged the northern sky every few nights since 2018
* we have 57,546 ZTF light curves of the field around M31 (Andromeda), mostly stars, with no labels - we want the odd ones out (outliers, or anomalies)
* each is described by 42 features (amplitude, skewness, periodogram peaks, ...) computed by SNAD, a team that searches ZTF for anomalies

In [ ]:
# four typical M31-field light curves, by anomaly rank (of 57,546)
plot_m31_typical()

<div style="font-size:0.8em; text-align:center; color:#666;">ZTF light curves from IRSA. Features from SNAD (Malanchev et al. 2021, MNRAS 502, 5147).</div>

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**An outlier is an object with few neighbors in feature space.**

</div>

## An isolation forest isolates each object with random cuts

In [ ]:
# 250 points in 2-D: random cuts until the red star is alone in its box
isolation_controls()

<img src="images/day22_static_isolation_widget.png" alt="The isolation widget after all cuts: a typical point at the center of a round cloud of 250 points needs 13 random cuts before it is alone in its box, an outlier at the lower right needs 3" style="max-height:330px; display:block; margin:0 auto;">

(static copy of the widget, for readers on GitHub or without running the notebook)

* a toy: 250 points with two made-up features (M31's light curves have 42), and each cut picks a feature and a place at random
* slide the cuts up from 0 - the outlier is alone after 3, the typical point after 13

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**An isolation forest is unsupervised (unlike a random forest) because random cuts need no labels.**

</div>

## Fewer cuts mean a higher anomaly score

In [ ]:
# 200 random trees on the same 250 points: how many cuts each one needs to isolate the two red stars
plot_depths()

* one tree can get lucky, so we do the thing we always do - grow many and average the number of cuts
* the anomaly score runs from 0 to 1, and 0.5 is a typical object

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**An object that most trees isolate in a few cuts gets a high anomaly score.**

</div>

## The forest scores M31's top outliers only about 0.6

* back to the 57,546 M31 light curves: slide through ranks 1 to 12, then 100, 1,000, 10,000, the median and 57,000
* scores bunch near 0.5, so 0.62 is the far tail, and the shaded months are the data the features came from

In [ ]:
# isolation forest on 57,546 light curves (1,000 trees, each feature replaced by its rank; log scale)
m31_controls()

<img src="images/day22_static_m31_widget.png" alt="The M31 widget at rank 1: the histogram of 57,546 anomaly scores with a red line at 0.62, and the rank-1 light curve, steady at 15.5 mag with three faint points in 2018" style="max-height:330px; display:block; margin:0 auto;">

(static copy of the widget, for readers on GitHub or without running the notebook)

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**You have to look at each light curve to know why it's odd.**

</div>

## The top outlier looks like bad data

In [ ]:
# ZTF r-band light curves from IRSA, NASA's archive that serves ZTF data (shaded: the features' months)
plot_m31_lcs()

* rank 1 is a steady 15.5-magnitude star with three points much fainter in August 2018, almost surely bad data
* ranks 2 and 6 look like real variable stars, and rank 28,773 is a typical star

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**It's your job to decide whether an odd light curve is interesting.**

</div>

## Two-thirds of SNAD's top outliers were bad data

* SNAD searched 2.25 million ZTF light curves in three fields, M31's among them, with four outlier algorithms (one an isolation forest)
* an expert then checked the top 277 - the bad data came from errors in image subtraction, and from stars that overlapped known asteroids
* the 23 new ones include four supernova candidates and a flare on a red dwarf star

In [ ]:
# what an expert found among the 277 top outliers
plot_snad()

<div style="font-size:0.8em; text-align:center; color:#666;">Malanchev, Pruzhinskaya, Korolev et al. 2021, MNRAS 502, 5147 (arXiv:2012.01419).</div>

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Look at your outliers (most of the top ones are bad data)!**

</div>

## Can a method find groups with no labels?

* the isolation forest found the odd ones out - now we look for groups
* we switch to 2,884 SDSS stars, galaxies and quasars, because their spectra tell us afterward which is which
* a color is a difference of two magnitudes (SDSS filters u, g, r, i, z), and a small $u - g$ is blue

<div style="font-size:0.8em; text-align:center; color:#666;">SDSS objects from the ASTR 496 (Fall 2025) week 12 files.</div>

In [ ]:
# the SDSS objects in two colors, with no labels
plot_sdss_gray()

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Can you see three groups here?**

</div>

## k-means is the most basic approach to finding clusters

* assign each point to the nearest of $k$ centers, move each center to the mean of its points, repeat ($k$ counts clusters, not Tuesday's neighbors)
* slide $k$ from 1 to 8 (clusters and their centers on the left, classes from the spectra on the right)

In [ ]:
# k-means on the two colors (left), next to the classes from the spectra (right)
kmeans_controls()

<img src="images/day22_static_kmeans_widget.png" alt="The k-means widget at k = 3: left, the objects colored by cluster with three black crosses at the centers - one on the quasars at small u - g, two along the band of galaxies and stars; right, the same points colored by spectroscopic class, with the stars along a long diagonal band" style="max-height:330px; display:block; margin:0 auto;">

(static copy of the widget, for readers on GitHub or without running the notebook)

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**k-means always returns k clusters, whether or not the data have k classes.**

</div>

## k-means with k = 3 finds the quasars

In [ ]:
# how many of each spectroscopic class landed in each k-means cluster (k = 3)
plot_kmeans_table()

* the cyan cluster is 91% quasars
* the stars are split between the other two clusters, with the galaxies

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**The olive and brown clusters are each about half stars and half galaxies.**

</div>

## A Gaussian mixture lets each cluster stretch

In [ ]:
# left: three Gaussians fit by EM to two colors; right: BIC for 1 to 12 Gaussians in all four colors
plot_gmm()

* a **Gaussian mixture model** describes the data as a sum of Gaussians, and k-means is the special case with round ones
* fit it by repeating two steps - give each point a probability for each Gaussian, then refit them (**expectation-maximization**, EM)
* BIC (Day 8, lower is better) on four colors ($u-g$, $g-r$, $r-i$, $i-z$) is still falling at 12 Gaussians

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**BIC is still falling at 12 Gaussians for only 3 classes.**

</div>

## Is a galaxy spectrum really 1,000 independent numbers?

* the classes we just checked came from spectra, and a spectrum is 1,000 numbers, not 2 or 4 colors
* 968 SDSS galaxy spectra, each the flux at 1,000 rest wavelengths
* the flux (at least at nearby wavelengths) is *correlated*

In [ ]:
# four of the galaxies, offset for clarity, with four emission lines marked
plot_spectra()

<div style="font-size:0.8em; text-align:center; color:#666;">SDSS galaxy spectra from astroML's spec4000 sample (VanderPlas & Connolly 2009).</div>

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**We shouldn't need 1,000 numbers to describe a spectrum that's roughly a smooth continuum plus some lines.**

</div>

## Principal component analysis (PCA) rotates the axes

* in 2-D: the stars' band in two colors, with the direction they spread most along (component 1) and the one across it (component 2)

In [ ]:
# the SDSS stars in two colors, and the same stars with the axes rotated onto the components
plot_pca_2d()

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**PCA rotates the axes to line up with the directions the data spread along.**

</div>

## Each component of a galaxy spectrum is itself a spectrum

* the same rotation in 1,000 dimensions: the components are the eigenvectors of the covariance of the fluxes at the 1,000 wavelengths

In [ ]:
# the mean and first three components (left), and the variance that n components explain
plot_eigenspectra()

* each galaxy = mean + $a_1 \times$ component 1 + $a_2 \times$ component 2 + ... (each eigenvalue is the variance of its $a$)

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Describing each galaxy with a few components is called dimensionality reduction (the first explains 57% of the variance, and five explain 88%).**

</div>

## A few components reconstruct a galaxy's continuum

In [ ]:
# slide n: one galaxy, and the mean spectrum plus its first n components
reconstruct_controls()

<img src="images/day22_static_pca_widget.png" alt="The PCA widget at 2 components: one galaxy spectrum in gray and the mean plus two principal components in red, which follows the continuum and the [OII] and [OIII] lines but falls short of the tall H-alpha line" style="max-height:330px; display:block; margin:0 auto;">

(static copy of the widget, for readers on GitHub or without running the notebook)

* this galaxy has strong emission lines - slide from 0 (the mean spectrum) to 2, 5, 10 and 30, and look at H$\alpha$ near 6563 Angstrom

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**The tall H$\alpha$ line needs the most components.**

</div>

## The first component partly separates the star-forming galaxies

In [ ]:
# each galaxy's first two PCA coefficients (blue: star-forming emission lines, from SDSS)
plot_pc12()

* star-forming galaxies have young, hot stars whose ultraviolet light ionizes gas, which then glows in emission lines like H$\alpha$ - we colored them blue after PCA ran
* remember that PCA is a linear method - curved structure (like the stars in color space) needs non-linear methods

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**PCA partly separates the star-forming galaxies without ever seeing a label.**

</div>

## Galaxies at the two ends of component 1 look different

* the three galaxies with the most negative and the most positive component 1

In [ ]:
# spectra at the two ends of component 1, offset for clarity
plot_pc1_ends()

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Look at the spectra at the extremes before you name a component.**

</div>

## Hand-picked features are what the bitter lesson warns about

* "The biggest lesson that can be read from 70 years of AI research is that general methods that leverage computation are ultimately the most effective, and by a large margin." (Sutton 2019)
* our expertise helps first - Tuesday's 84 features couldn't even be computed for a kilonova
* next Tuesday's neural networks learn the features themselves, and you still have to look at what they found

In [ ]:
# what we chose, and what the method learned, for the classifiers in this course
plot_ladder()

<div style="font-size:0.8em; text-align:center; color:#666;">R. Sutton, "The Bitter Lesson" (2019, incompleteideas.net); the point as Josh Speagle (Toronto) made it in his SkAI colloquium.</div>

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Sutton argues that methods that learn from data beat the features we choose.**

</div>

## What did we learn today?

* rare classes: at 1%, a 99% accurate forest missed 95% of the kilonovae
* real time: in their first 3 days, superluminous supernovae barely stand out
* missing data: dropping the faint supernovae left a forest that got 60% of them right
* no labels: the top M31 outlier was bad data, k-means always returns k clusters, and a few PCA components describe a spectrum
* next Tuesday: neural networks, which learn the features themselves

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**LOOK AT YOUR DATA before you believe what any method finds, with or without labels!**

</div>

## Before you go

* Lab 09 (ABC on a contaminated Hubble diagram) is due **Tue Nov 17 by Noon**
* notes on Jessie Muir's colloquium (a page, to make up lost Quiz 4 points) are due **Fri Nov 13**
* Lab 10 (machine learning: rare classes, missing data, clustering) posts Thu Nov 19, due Wed Dec 2
* Tue Nov 17 is on Zoom (I'm traveling): neural networks
* colloquium Tue Nov 17, 3:45 pm, 134 Astronomy: Shazrene Mohamed (Virginia)
* every date is in `ASSIGNMENTS.md`

## Lab 08 defenses

* **Celeste Emery** and **Ben Hall-Skank**, up front together
* Abha and I ask questions of both of you, about Lab 08
* everyone else stays and listens